# Legacy notebook

Contains broken file paths to legacy versions of files.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import pandas as pd
from IPython.display import display
import yaml
from conlanger.tools.rules import DiachronicSeries, DebugRules
from conlanger.appliers.asca import run_asca
from conlanger.appliers.brassica import run_brassica

RULES_DIR = "../data/{format}/rules"
DEBUG_RULES_DIR = "../data/{format}/rules/tmp"
OVERWRITE_RULES = True
CREATE_DEBUG_RULES = True
RULES_YAML = "./data/index_diachronica.yml"
RULES_YAML = "./data/index_diachronica_ai.yml"

In [ ]:
asca_debug_results = []
asca_debug_word_file = "../data/generated/lexicon/asca/weirdness_0.5.wsca"
asca_debug_rule_path = f"{DEBUG_RULES_DIR.format(format="asca")}"
syntax_set = set()

if CREATE_DEBUG_RULES:
    with open(RULES_YAML, "r") as f:
        doc = yaml.load(f, Loader=yaml.SafeLoader)
    sections = doc['sections']

    print(f"sections: {len(sections)}")
    
    debug_rules = []

    for section in sections:
        index = section["index"]
        if len(section.get("rules", [])) > 0:
            for i, r in DebugRules(section, format="asca"):
                syntax = str(r._parts[1]).strip()
                if syntax not in syntax_set:
                    syntax_set.add(syntax)
                    debug_rules.append((i, (r.title + '.rsca').replace(' - ', '_'), str(r), syntax))

    for i, name, rule, syntax in debug_rules:
        file_name = f"{asca_debug_rule_path}/{name}"
        with open(file_name, "w") as f:
            f.write(rule)

        result = run_asca(asca_debug_word_file, name, asca_debug_rule_path)
        asca_debug_results.append({**result, "syntax": syntax})

        os.remove(file_name)

    asca_debug_results_df = pd.DataFrame(asca_debug_results)

    print(asca_debug_results_df["returncode"].value_counts())

    errors = asca_debug_results_df[asca_debug_results_df["returncode"] != 0][["rule", "error", "syntax"]]
    errors = errors[errors["error"] != "Runtime Error: Can't delete a word's only segment"]
    errors["section"] = errors["rule"].str.split("_").str[0]
    errors["idx"] = errors["rule"].str.split("_").str[1].str.split(".").str[0]

    errors[["section", "idx", "syntax", "error"]].to_csv("../data/asca/results/asca_errors.csv", index=False)

    error_counts = errors["error"].value_counts().to_frame(name="count").reset_index()
    display(error_counts.head(10))
    error_counts.to_csv("../data/asca/results/asca_error_counts.csv", index=True)

    unknown_groupings = errors[errors["error"].str.contains("Unknown grouping")]
    unknown_groupings["grouping"] = unknown_groupings["error"].str.replace("Syntax Error: Unknown grouping '", "").str[0]
    unknown_groupings = unknown_groupings[["grouping", "syntax", "section", "idx"]]
    display(unknown_groupings)
    


    for r in errors[:20].itertuples(index=False):
        print(r.section, r.idx, r.syntax, r.error)
                

In [ ]:
if OVERWRITE_RULES:
    with open(RULES_YAML, "r") as f:
        doc = yaml.load(f, Loader=yaml.SafeLoader)
    sections = doc['sections']

    results = []
    rules = []

    for section in sections:
        index = section["index"]
        results.append({
            "index": index,
            "name": section["section"],
            "rule_count": len(section.get("rules", [])),
        })

        rules.append((index, str(DiachronicSeries(section, format="asca"))))


    sections_df = pd.DataFrame(results)

    sections_df.to_csv("../data/diachronica/index_diachronica_sections.csv", index=False)
    
    for index, rule in rules:
        with open(f"../data/asca/rules/{index}.rsca", "w") as f:
            f.write(rule)

    print(sections_df.head(3))

In [ ]:
sections_df = pd.read_csv("../data/diachronica/index_diachronica_sections.csv", dtype={"index": str, "name": str, "rule_count": int})

rules_df = sections_df[sections_df["rule_count"] > 0].copy()
rules_df['asca_rule_file'] = rules_df['index'].apply(lambda x: f"{x}.rsca")
rules_df['brassica_rule_file'] = rules_df['index'].apply(lambda x: f"{x}.bsc")

display(rules_df.head(3))

asca_rule_files = rules_df['asca_rule_file'].tolist()
brassica_rule_files = rules_df['brassica_rule_file'].tolist()


In [ ]:
# run asca-rust to validate rules

asca_results = []
asca_word_file = "../data/generated/lexicon/asca/weirdness_0.5.wsca"
asca_alias_file = "../data/asca/asca_aliases.alias"
asca_rule_path = f"{RULES_DIR.format(format="asca")}"

for rule_file in asca_rule_files[:100]:
    result = run_asca(asca_word_file, rule_file, asca_rule_path)
    asca_results.append(result)

asca_results_df = pd.DataFrame(asca_results)

asca_results_df.to_csv("../data/asca/results/asca_results.csv", index=False)

print(asca_results_df["returncode"].value_counts())

errors = asca_results_df[asca_results_df["returncode"] != 0][["rule", "error"]]

print(f"errors: {len(errors)}")

for r in errors[:20].itertuples(index=False):
    print(r.rule, r.error)

In [ ]:
# run Brassica to validate rules

# brassica_results = []
# brassica_word_file = "../data/generated/lexicon/brassica/weirdness_0.5.lex"

# for rule_file in brassica_rule_files[:10]:
#     result = run_brassica(brassica_word_file, rule_file)
#     brassica_results.append(result)

# brassica_results_df = pd.DataFrame(brassica_results)

# brassica_results_df.to_csv("../data/brassica/results/brassica_results.csv", index=False)

# brassica_results_df[brassica_results_df["returncode"] != 0].head()